# DuckDB and file formats - homework

1. Download data:

All the data are downloaded and stored in data/csv

In [ ]:
import os
import subprocess

DATA_DIR = "data"
if not os.path.exists(DATA_DIR):
    os.mkdir(DATA_DIR)

DATA_DIR = "data"
CSV_DIR = os.path.join(DATA_DIR, "csv")

if not os.path.exists(CSV_DIR):
    os.makedirs(CSV_DIR)

distances_url = "https://opendata.rijdendetreinen.nl/public/tariff-distances/tariff-distances-2022-01.csv"
distances_path = os.path.join(CSV_DIR, "tariff-distances-2022-01.csv")

if not os.path.exists(distances_path):
    subprocess.run(["wget", distances_url, "-O", distances_path])

total_size = os.path.getsize(distances_path)
total_size_mb = total_size // (1024 * 1024)
print(f"Total dataset size: {total_size_mb} MB")

In [ ]:
total_size = 0

stations_url = "https://opendata.rijdendetreinen.nl/public/stations/stations-2023-09-nl.csv"
stations_path = os.path.join(CSV_DIR, "stations-2023-09-nl.csv")

if not os.path.exists(stations_path):
    subprocess.run(["wget", stations_url, "-O", stations_path])

total_size = os.path.getsize(stations_path)
total_size_mb = total_size // (1024 * 1024)
print(f"Total dataset size: {total_size_mb} MB")

In [ ]:
total_size = 0
years = range(2011, 2024)

for year in years:
    filename = f"disruptions-{year}.csv"
    filepath = os.path.join(CSV_DIR, filename)
    if not os.path.exists(filepath):
        subprocess.run(
            [
                "wget",
                f"https://opendata.rijdendetreinen.nl/public/disruptions/{filename}",
                "-O",
                filepath,
            ]
        )
    else:
        print(f"File {filename} already exists.")

    if os.path.exists(filepath):
        total_size += os.path.getsize(filepath)

total_size_mb = total_size // (1024 * 1024)
print(f"Total dataset size: {total_size_mb} MB")

In [ ]:
years = range(2019, 2022)
total_size = 0

for year in years:
    filename = f"services-{year}.csv.gz"
    filepath = os.path.join(CSV_DIR, filename)
    url = f"https://opendata.rijdendetreinen.nl/public/services/{filename}"

    if not os.path.exists(filepath):
        subprocess.run(["wget", url, "-O", filepath])
    else:
        print(f"File {filename} already exists.")

    if os.path.exists(filepath):
        total_size += os.path.getsize(filepath)

total_size_mb = total_size // (1024 * 1024)
print(f"Total dataset size: {total_size_mb} MB")

2. Put stations data into `stations` table in DuckDB. This changes rarely, so we treat it as a almost constant file.


In [ ]:
import duckdb

db = duckdb.connect("data/duckdb_trains.db")

db.sql("""
           CREATE TABLE IF NOT EXISTS stations AS
               FROM 'data/csv/stations-2023-09-nl.csv';
           """)

In [ ]:
db.sql("""
       SELECT id,
              name_short,
              name_long,
              country,
              printf('%.2f', geo_lat) AS latitude,
              printf('%.2f', geo_lng) AS longitude
       FROM stations LIMIT 5;
       """).show()

3. Based on [DuckDB tutorial](https://duckdb.org/2024/05/31/analyzing-railway-traffic-in-the-netherlands.html#largest-distance-between-train-stations-in-the-netherlands),
   create tables `distances` and `distances_long`. We treat this similarly to `stations` table.

In [ ]:
db.sql("""
CREATE TABLE IF NOT EXISTS distances AS
FROM 'data/csv/tariff-distances-2022-01.csv';

""")

In [ ]:
db.sql("""
       SELECT Station,
              AC,
              AH,
              ALM
       FROM distances LIMIT 5;
       """).show()

In [ ]:
db.sql("""
       CREATE TABLE IF NOT EXISTS distances_long AS
           UNPIVOT distances
           ON COLUMNS
       (
           *
           EXCLUDE
       (
           Station
       ))
           INTO
           NAME destination
           VALUE distance;
       """)


In [ ]:
db.sql("""
       SELECT Station     AS start_station,
              destination AS end_station,
              distance
       FROM distances_long LIMIT 5;
       """).show()

4. Put train disruptions into `disruptions` table in the Postgres database. We expect this data to change
   regularly, and thus treat it as a typical OLTP table.

In [ ]:
db.sql("INSTALL postgres; LOAD postgres;")

db.sql(
    "ATTACH IF NOT EXISTS 'dbname=postgres user=postgres password=postgres host=localhost' AS postgres_db (TYPE POSTGRES);")

db.sql("""
       DROP TABLE IF EXISTS postgres_db.disruptions;
       CREATE TABLE postgres_db.disruptions AS
       SELECT *
       FROM read_csv_auto('data/csv/disruptions-*.csv');
       """)


In [ ]:
db.sql("""
       SELECT COUNT(*)
       FROM postgres_db.disruptions
       """).show()

In [ ]:
db.sql("""
       SELECT *
       FROM postgres_db.disruptions LIMIT 5;
       """).show()

5. Transform train services CSV files into a single Parquet file. Make table `services` from it. We treat
   this as a big data batch input, created rarely but regularly for analytics purposes.

In [ ]:
db.sql("""
       COPY
       (
       SELECT *
       FROM read_csv('data/csv/services*.csv.gz',
                     header = True,
                     all_varchar = False,
                     ignore_errors = True) ) TO 'data/services.parquet' (FORMAT 'PARQUET');
       """)

db.sql("""
CREATE TABLE IF NOT EXISTS services AS
SELECT * FROM 'data/services.parquet';
""")

result = db.sql("SELECT count(*) FROM services").fetchone()

6 & 7. Make queries to answer the following questions:

7.1. How many trains departed from Amsterdam Central station overall?

In [ ]:
db.sql("""
       SELECT count(*)
       FROM services
       WHERE "Stop:Arrival time" IS NULL
         AND "Stop:Station code" LIKE 'AS%'
       """)



7.2 Calculate the average arrival delay of different service types (`Service:Type`). Order results descending by average delay.


In [ ]:
db.sql("""
       SELECT "Service:Type",
              AVG("Stop:Arrival delay") AS avg_arrival_delay
       FROM services
       WHERE "Stop:Arrival delay" IS NOT NULL
       GROUP BY "Service:Type"
       ORDER BY avg_arrival_delay DESC
       """).show()

   7.3 What was the most common disruption cause in different years? [MODE function](https://duckdb.org/docs/stable/sql/functions/aggregates.html#modex) may be useful.


In [ ]:
db.sql("""
       SELECT EXTRACT(year FROM CAST(start_time AS TIMESTAMP)) AS disruption_year,
              MODE(cause_en)                                   AS most_common_cause
       FROM postgres_db.disruptions
       GROUP BY disruption_year
       ORDER BY disruption_year;
       """).show()

   7.4 How many trains started their overall service in any Amsterdam station?

In [ ]:
db.sql("""
       SELECT count(*)
       FROM services
       WHERE "Stop:Arrival time" IS NULL
         AND "Stop:Station code" IN (SELECT code
                                     FROM stations
                                     WHERE name_long LIKE 'Amsterdam%')
       """).show()

   7.5 What fraction of services was run to final destinations outside the Netherlands?


In [ ]:
db.sql("""
       SELECT AVG((st.country != 'NL')::INT) AS fraction_outside_nl
       FROM services s
                JOIN stations st ON s."Stop:Station code" = st.code
       WHERE s."Stop:Departure time" IS NULL;
       """).show()

   7.6 What is the largest distance between stations in the Netherlands (code `NL`)?


In [ ]:
db.sql("""
       SELECT MAX(d.distance) AS max_distance
       FROM distances_long d
                JOIN stations s1 ON d.Station = s1.code
                JOIN stations s2 ON d.destination = s2.code
       WHERE s1.country = 'NL'
         AND s2.country = 'NL'
         AND d.distance != 'XXX';
       """).show()

7.7. Compare the average arrival delay between different train operators (`Service:Company`) on a bar plot. Sort them appropriately.

In [ ]:
df = db.sql("""
            SELECT "Service:Company", AVG("Stop:Arrival delay") as avg_delay
            FROM services
            WHERE "Stop:Arrival delay" IS NOT NULL
            GROUP BY ALL
            ORDER BY avg_delay DESC
            """).df()

df.plot.bar(x='Service:Company', y='avg_delay', title='Average Arrival Delay by Company', ylabel='Minutes', rot=45)

7.8. How many services were disrupted in different years? Make a line plot.

In [ ]:
services_disrupted = db.sql("""
                            SELECT EXTRACT(year FROM CAST("Service:Date" AS DATE)) AS year,
        COUNT(DISTINCT "Service:RDT-ID") AS disrupted_services_count
                            FROM services
                            WHERE "Service:Maximum delay"
                                > 0
                               OR "Service:Completely cancelled" = True
                               OR "Service:Partly cancelled" = True
                            GROUP BY year
                            ORDER BY year
                            """).df()

services_disrupted.plot.line(
    x='year',
    y='disrupted_services_count',
    title='Disrupted Services per Year',
    xlabel='Year',
    ylabel='Disrupted Services'
)

7.9 What fraction of all services were cancelled (`Service:Completely cancelled`) in different years? Make a line plot.


In [ ]:
services_cancelled = db.sql("""
                            SELECT EXTRACT(year FROM CAST("Service:Date" AS DATE)) AS year,
        AVG(CAST("Service:Completely cancelled" AS INTEGER)) AS fraction_cancelled
                            FROM services
                            GROUP BY year
                            ORDER BY year
                            """).df()

services_cancelled.plot.line(
    x='year',
    y='fraction_cancelled',
    title='Cancelled Services per Year',
    xlabel='Year',
    ylabel='Cancelled Services'
)

8. Currently, `services` table does not provide information about service lengths, neither between pairs of stations
   nor for the end-to-end service. Prepare this information:
   1. Note that each service has the same `Service:RDT-ID`, and stations can be ordered by `Stop:Departure time`,
      with the last one being NULL. Using [window functions](https://duckdb.org/docs/stable/sql/functions/window_functions.html),
      specifically [LAG() or LEAD()](https://duckdb.org/docs/stable/sql/functions/window_functions.html),
      you can get next row. [This example](https://stackoverflow.com/a/62584847/9472066) may also be useful.
   2. Create table `station_connections`, with columns `Service:RDT-ID`, `start_station_code` and `end_station_code`
      (pair of stations on a route), and `distance` between them. Note that you should deduplicate the data on station
      codes, so that every station pair appears only once. Create temporary tables, use a subquery, or any other
      similar techniques if necessary.
   3. What is the largest distance between a pair of stations?
   4. Plot a histogram of inter-station distances run by trains.

In [ ]:
db.sql("""
CREATE OR REPLACE TABLE station_connections AS
WITH segments AS (
    SELECT
        "Stop:Station code" AS start_station_code,
        LEAD("Stop:Station code") OVER (
            PARTITION BY "Service:RDT-ID"
            ORDER BY "Stop:Departure time" ASC
        ) AS end_station_code
    FROM services
)
SELECT DISTINCT
    s.start_station_code,
    s.end_station_code,
    CAST(d.distance AS INTEGER) AS distance
FROM segments s
JOIN distances_long d
  ON s.start_station_code = d.Station
  AND s.end_station_code = d.destination
WHERE s.end_station_code IS NOT NULL
  AND d.distance != 'XXX';
""")

db.sql("SELECT MAX(distance) AS max_interstation_distance FROM station_connections").show()

import matplotlib.pyplot as plt

df_distances = db.sql("SELECT distance FROM station_connections").df()
df_distances['distance'].plot.hist(
    bins=30,
    title='Distance between stations',
    edgecolor='black'
)
plt.xlabel('Distance (km)')
plt.ylabel('Frequency')
plt.show()